# 数据集加载及预处理

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
import numpy as np

# ==============================
# 加载加利福尼亚房价数据集
# ==============================
# fetch_california_housing 会自动从网上下载数据（如已下载会直接读取）
# data_home='./data' 指定数据保存路径
california_housing = fetch_california_housing(data_home='./data')
X, y = california_housing.data, california_housing.target  # X是特征, y是房价标签(单位: 千美元)

# ==============================
# 划分数据集：训练、验证、测试
# ==============================
from sklearn.model_selection import train_test_split

# Step 1：先划分训练集（70%），剩余30%留作临时集(后续再分验证/测试)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42    # random_state保证可复现
)

# Step 2：把临时集一分为二，分别为验证集和测试集（各占15%）
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)
# 此时训练/验证/测试比例约为70%/15%/15%

# ==============================
# 数据标准化处理
# 仅对特征X进行标准化，不对标签y标准化
# ==============================
# 标准化操作：减均值/除标准差，常用于提升神经网络收敛速度
scaler_X = StandardScaler()

# 训练集用fit_transform，计算统计参数并变换
X_train_scaled = scaler_X.fit_transform(X_train)
# 验证集/测试集仅用transform（使用训练集统计参数）
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

# ==============================
# 封装Dataset，便于与PyTorch DataLoader配合使用
# ==============================
class CaliforniaHousingDataset(Dataset):
    def __init__(self, X, y):
        """
        初始化数据集
        参数:
            X: 标准化后的特征(ndarray)
            y: 原始目标标签(ndarray)
        """
        self.X = torch.FloatTensor(X)      # 转为torch张量(float32)
        self.y = torch.FloatTensor(y)      # 目标也是float32回归数据

    def __len__(self):
        """
        返回数据集样本数
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        支持索引，返回第idx个样本（特征/标签）
        标签增加一个维度，输出形状为[1]
        """
        return self.X[idx], self.y[idx].unsqueeze(0)
        # 这里y加unsqueeze(0), 变成1维, 适配回归输出形状(batch, 1)

# ==============================
# 构造PyTorch数据集对象
# ==============================
train_dataset = CaliforniaHousingDataset(X_train_scaled, y_train)
val_dataset = CaliforniaHousingDataset(X_val_scaled, y_val)
test_dataset = CaliforniaHousingDataset(X_test_scaled, y_test)

# ==============================
# DataLoader：小批量加载数据，支持打乱/多线程
# ==============================
batch_size = 64   # 批大小(64条样本一个batch)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)    # 训练集打乱
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)       # 验证/测试不打乱
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==============================
# 打印数据集相关信息，便于核查
# ==============================
print(f"数据集大小: 总计 {len(X)}")
print(f"训练集: {len(train_dataset)} 样本")
print(f"验证集: {len(val_dataset)} 样本")
print(f"测试集: {len(test_dataset)} 样本")
print(f"特征维度: {X.shape[1]}")


数据集大小: 总计 20640
训练集: 14448 样本
验证集: 3096 样本
测试集: 3096 样本
特征维度: 8


In [3]:
for x,y in train_loader:
    print(x.shape)
    print(y.shape)
    break

torch.Size([64, 8])
torch.Size([64, 1])


In [4]:


# 演示批量情况
print(f"\n批量情况演示:")
y_batch = torch.tensor([1.2, 2.3, 3.4])  # 3个样本的标签
print(f"批量标签原始形状: {y_batch.shape}")
y_batch_unsqueezed = y_batch.unsqueeze(1)  # 在第1维增加维度
print(f"批量标签unsqueeze(1)后形状: {y_batch_unsqueezed.shape}")
print(f"批量标签内容:\n{y_batch_unsqueezed}")



批量情况演示:
批量标签原始形状: torch.Size([3])
批量标签unsqueeze(1)后形状: torch.Size([3, 1])
批量标签内容:
tensor([[1.2000],
        [2.3000],
        [3.4000]])


# 搭建模型

In [5]:
# 定义回归模型
class HousePriceModel(nn.Module):
    def __init__(self, input_size, hidden_size=30, output_size=1):
        super(HousePriceModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 创建模型实例
input_size = X.shape[1]  # 特征维度
model = HousePriceModel(input_size=input_size, hidden_size=30, output_size=1)

print(f"模型结构:")
print(model)
print(f"\n模型参数数量: {sum(p.numel() for p in model.parameters())}")


模型结构:
HousePriceModel(
  (fc1): Linear(in_features=8, out_features=30, bias=True)
  (fc2): Linear(in_features=30, out_features=1, bias=True)
  (relu): ReLU()
)

模型参数数量: 301


In [6]:
# 导入训练模块
from wangdao_train import Trainer, EarlyStopping, ModelCheckpoint


# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 初始化损失函数和优化器
criterion = nn.MSELoss()  # 均方误差损失函数，适用于回归任务
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 初始化早停对象
early_stopping = EarlyStopping(
    patience=10,      # 容忍10个评估周期没有改善
    min_delta=0.001,  # 最小改善阈值
    mode='min'        # 监控val_loss，越小越好
)

# 初始化模型保存对象
model_checkpoint = ModelCheckpoint(
    filepath='./checkpoints/regression_model_epoch_{epoch}.ckpt',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    min_delta=0.001
)

# 创建训练器
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=50,  # 每50个batch评估一次
    early_stopping=early_stopping,
    model_checkpoint=model_checkpoint
)

# 开始训练
print("开始训练...")
trainer.train_regression(num_epochs=100)


使用设备: cpu
开始训练...
[Step 50] Val Loss: 4.9131
[Step 100] Val Loss: 3.0643
[Step 150] Val Loss: 1.7876
[Step 200] Val Loss: 1.1456
Epoch [1/100]  Train Loss: 3.1851
[Step 250] Val Loss: 0.9142
[Step 300] Val Loss: 0.8387
[Step 350] Val Loss: 0.7956
[Step 400] Val Loss: 0.7568
[Step 450] Val Loss: 0.7250
Epoch [2/100]  Train Loss: 0.7969
[Step 500] Val Loss: 0.6923
[Step 550] Val Loss: 0.6601
[Step 600] Val Loss: 0.6322
[Step 650] Val Loss: 0.6053
Epoch [3/100]  Train Loss: 0.6316
[Step 700] Val Loss: 0.5814
[Step 750] Val Loss: 0.5620
[Step 800] Val Loss: 0.5444
[Step 850] Val Loss: 0.5290
[Step 900] Val Loss: 0.5166
Epoch [4/100]  Train Loss: 0.5330
[Step 950] Val Loss: 0.5063
[Step 1000] Val Loss: 0.4981
[Step 1050] Val Loss: 0.4889
[Step 1100] Val Loss: 0.4829
Epoch [5/100]  Train Loss: 0.4800
[Step 1150] Val Loss: 0.4759
[Step 1200] Val Loss: 0.4716
[Step 1250] Val Loss: 0.4694
[Step 1300] Val Loss: 0.4619
[Step 1350] Val Loss: 0.4583
Epoch [6/100]  Train Loss: 0.4523
[Step 1400] Val